# CSIL / CSIL+SOAR resume — Pendulum sweep

Resumes the missing CSIL and CSIL+SOAR Pendulum runs using the fixed hyperparameters from the `fix-csil-pendulum` branch:
- `scale_factor = 0.1` (was 1.0 — too aggressive on Pendulum)
- `bc_steps = 5000` (was 20000 — caused BC overfit on small K)
- `total_steps = 30000` (was 50000 — peak is around step 25k)

**Setup (one-time):**
1. Upload these two files to your Google Drive folder `MyDrive/imitation_learning/`:
   - `src/csil.py` (replace the old one)
   - `resume_csil.py` (new file)
2. Open this notebook in Colab.
3. Runtime → Change runtime type → **CPU** (GPU won't help here).
4. Run the cells in order.

**Estimated wall-clock:** ~2.5 hours for the 11 missing Pendulum runs.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Move into the project folder

In [ ]:
%cd /content/drive/MyDrive/imitation_learning
!ls resume_csil.py src/csil.py

Both files should appear in the listing with today's timestamp. If `resume_csil.py` is missing or `src/csil.py` has an old date → re-upload.

## 3. Install dependencies (gymnasium only)

In [ ]:
!pip install gymnasium -q

## 4. Sanity check that the new config is loaded

Expected output: `scale_factor: 0.1`, `bc_steps: 5000`, `total_steps: 30000` for Pendulum.

If you see `bc_steps: 20000` or no `scale_factor` key → the file in Drive is the old version.

In [ ]:
import sys
sys.path.insert(0, 'src')
from csil import ENV_CONFIGS
print('CartPole:', ENV_CONFIGS['CartPole'])
print('Pendulum:', ENV_CONFIGS['Pendulum'])

## 5. See what's already done and what's missing

Quick inventory before kicking off the resume.

In [ ]:
import os

K_VALUES = [1, 3, 5, 10, 15]
SEEDS = [42, 43, 44]

for algo in ['csil', 'csilsoar']:
    done, missing = [], []
    for K in K_VALUES:
        for seed in SEEDS:
            f = f'logs/{algo}_Pendulum_K{K}_seed{seed}.csv'
            (done if os.path.exists(f) else missing).append((K, seed))
    print(f'{algo.upper()}: {len(done)}/15 done, {len(missing)} missing -> {missing}')

## 6. Run the resume (~2.5 hours)

Skips configs that already have CSVs. Saves to `logs/csil_Pendulum_K*_seed*.csv` and `logs/csilsoar_Pendulum_K*_seed*.csv` on Drive automatically.

In [ ]:
!python resume_csil.py

## 7. (Optional) Quick max-reward summary

In [ ]:
import csv
import numpy as np

for algo in ['csil', 'csilsoar']:
    print(f'\n=== {algo.upper()} Pendulum ===')
    for K in K_VALUES:
        maxes = []
        for seed in SEEDS:
            path = f'logs/{algo}_Pendulum_K{K}_seed{seed}.csv'
            if not os.path.exists(path):
                continue
            with open(path) as f:
                rdr = csv.DictReader(f)
                rewards = [float(r['eval_reward']) for r in rdr]
                maxes.append(max(rewards))
        if maxes:
            print(f'  K={K:<3} max={np.mean(maxes):.1f} (n={len(maxes)})')